In [1]:
import sys
print(sys.version)

3.10.19 (main, Oct  9 2025, 15:25:03) [Clang 16.0.0 (clang-1600.0.26.6)]


In [2]:
pip install tensorflow-macos tensorflow-metal pretty_midi numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.16.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
!pip install tqdm


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [6]:
import os
import pretty_midi
import numpy as np
import pickle
from tqdm import tqdm

In [11]:
import os

print(os.listdir("dataset"))

['maestro-v3.0.0', '.ipynb_checkpoints']


In [12]:
print(os.listdir("dataset/maestro-v3.0.0")[:5])

['maestro-v3.0.0.json', '2013', '2014', '2015', 'LICENSE']


In [13]:
DATASET_PATH = "dataset/maestro-v3.0.0"

In [14]:
midi_files = []

for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.endswith(".midi") or file.endswith(".mid"):
            midi_files.append(os.path.join(root, file))

print("Total MIDI files found:", len(midi_files))

Total MIDI files found: 1276


In [15]:
midi_files = midi_files[:400]
print("Using MIDI files:", len(midi_files))

Using MIDI files: 400


In [20]:
from tqdm import tqdm
import pretty_midi

all_notes = []

for file in tqdm(midi_files):
    try:
        midi = pretty_midi.PrettyMIDI(file)
        for instrument in midi.instruments:
            for note in instrument.notes:
                pitch = note.pitch
                raw_duration = note.end - note.start
                duration = round(raw_duration * 4) / 4.0
                duration = min(duration, 4.0)
                token = f"{pitch}_{duration}"
                token = f"{pitch}_{duration}"
                all_notes.append(token)
    except:
        continue

print("Total tokens extracted:", len(all_notes))

100%|█████████████████████████████████████████| 400/400 [00:41<00:00,  9.70it/s]

Total tokens extracted: 2065530


In [21]:
pitchnames = sorted(set(all_notes))
vocab_size = len(pitchnames)

note_to_int = {note: number for number, note in enumerate(pitchnames)}

print("Vocabulary size:", vocab_size)

Vocabulary size: 1301


In [22]:
sequence_length = 128

network_input = []
network_output = []

for i in range(0, len(all_notes) - sequence_length):
    seq_in = all_notes[i:i + sequence_length]
    seq_out = all_notes[i + sequence_length]

    network_input.append([note_to_int[note] for note in seq_in])
    network_output.append(note_to_int[seq_out])

print("Total sequences:", len(network_input))

Total sequences: 2065402


In [23]:
import numpy as np

network_input = np.array(network_input)
network_output = np.array(network_output)

print("Input shape:", network_input.shape)

Input shape: (2065402, 128)


In [24]:
split = int(0.9 * len(network_input))

X_train = network_input[:split]
X_val = network_input[split:]

y_train = network_output[:split]
y_val = network_output[split:]

print("Train:", X_train.shape)
print("Val:", X_val.shape)

Train: (1858861, 128)
Val: (206541, 128)


In [25]:
import os
import pickle

os.makedirs("data", exist_ok=True)

np.save("data/X_train.npy", X_train)
np.save("data/X_val.npy", X_val)
np.save("data/y_train.npy", y_train)
np.save("data/y_val.npy", y_val)

with open("data/vocab.pkl", "wb") as f:
    pickle.dump(pitchnames, f)

print("Preprocessing saved permanently ✅")

Preprocessing saved permanently ✅


In [26]:
max_sequences = 250000

X_train = X_train[:int(0.9 * max_sequences)]
y_train = y_train[:int(0.9 * max_sequences)]

X_val = X_val[:int(0.1 * max_sequences)]
y_val = y_val[:int(0.1 * max_sequences)]

print("New Train shape:", X_train.shape)
print("New Val shape:", X_val.shape)

New Train shape: (225000, 128)
New Val shape: (25000, 128)


In [27]:
np.save("data/X_train.npy", X_train)
np.save("data/X_val.npy", X_val)
np.save("data/y_train.npy", y_train)
np.save("data/y_val.npy", y_val)

print("Reduced dataset saved ✅")

Reduced dataset saved ✅


In [28]:
print(X_train.dtype)

int64


In [29]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [30]:
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim):
        super().__init__()
        self.token_embeddings = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length,
            output_dim=embed_dim
        )
        self.sequence_length = sequence_length

    def call(self, inputs):
        positions = tf.range(start=0, limit=self.sequence_length, delta=1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

In [33]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=None):  # ← FIXED HERE
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

In [34]:
sequence_length = 128
vocab_size = 1301

embed_dim = 256
num_heads = 4
ff_dim = 512

inputs = layers.Input(shape=(sequence_length,))
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(inputs)
x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)
x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)

x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(vocab_size, activation="softmax")(x)

model = keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_embedding_1          │ (None, 128, 256)       │       365,824 │
│ (PositionalEmbedding)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, 128, 256)       │     1,315,840 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ (None, 128, 256)       │     1,315,840 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1301)           │       334,357 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,331,861 (12.71 MB)

 Trainable params: 3,331,861 (12.71 MB)

 Non-trainable params: 0 (0.00 B)

In [35]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10


2026-02-22 02:13:30.725665: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


3516/3516 ━━━━━━━━━━━━━━━━━━━━ 1900s 539ms/step - accuracy: 0.0175 - loss: 5.1608 - val_accuracy: 0.0129 - val_loss: 5.0617
Epoch 2/10
3516/3516 ━━━━━━━━━━━━━━━━━━━━ 2379s 677ms/step - accuracy: 0.0179 - loss: 5.1336 - val_accuracy: 0.0191 - val_loss: 5.0480
Epoch 3/10
3516/3516 ━━━━━━━━━━━━━━━━━━━━ 1846s 525ms/step - accuracy: 0.0182 - loss: 5.1243 - val_accuracy: 0.0132 - val_loss: 5.0572
Epoch 4/10
3516/3516 ━━━━━━━━━━━━━━━━━━━━ 1791s 509ms/step - accuracy: 0.0184 - loss: 5.1279 - val_accuracy: 0.0170 - val_loss: 5.0375
Epoch 5/10
3516/3516 ━━━━━━━━━━━━━━━━━━━━ 1826s 519ms/step - accuracy: 0.0190 - loss: 5.0505 - val_accuracy: 0.0167 - val_loss: 5.0253
Epoch 6/10
3516/3516 ━━━━━━━━━━━━━━━━━━━━ 1821s 518ms/step - accuracy: 0.0197 - loss: 4.9600 - val_accuracy: 0.0202 - val_loss: 5.0192
Epoch 7/10
3516/3516 ━━━━━━━━━━━━━━━━━━━━ 1821s 518ms/step - accuracy: 0.0206 - loss: 4.9518 - val_accuracy: 0.0171 - val_loss: 4.9833
Epoch 8/10
3516/3516 ━━━━━━━━━━━━━━━━━━━━ 1820s 518ms/step - accur

In [36]:
import os
os.makedirs("models", exist_ok=True)

model.save("models/tuneformer_piano.h5")
print("Model saved successfully 🔥")

Model saved successfully 🔥


In [37]:
print("pitchnames:", 'pitchnames' in globals())
print("note_to_int:", 'note_to_int' in globals())
print("all_notes:", 'all_notes' in globals())

pitchnames: True
note_to_int: True
all_notes: True


In [38]:
import pickle

with open("data/vocab.pkl", "rb") as f:
    pitchnames = pickle.load(f)

int_to_note = {i: n for i, n in enumerate(pitchnames)}
note_to_int = {n: i for i, n in enumerate(pitchnames)}

vocab_size = len(pitchnames)
print("Vocab loaded ✅", vocab_size)

Vocab loaded ✅ 1301


In [40]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.position_embeddings = layers.Embedding(input_dim=sequence_length, output_dim=embed_dim)
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

    def call(self, inputs):
        positions = tf.range(start=0, limit=self.sequence_length, delta=1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def get_config(self):
        config = super().get_config()
        config.update({
            "sequence_length": self.sequence_length,
            "vocab_size": self.vocab_size,
            "embed_dim": self.embed_dim
        })
        return config


class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.rate = rate

    def call(self, inputs, training=None):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "ff_dim": self.ff_dim,
            "rate": self.rate
        })
        return config

In [41]:
from tensorflow.keras.models import load_model

model = load_model(
    "models/tuneformer_piano.h5",
    custom_objects={
        "PositionalEmbedding": PositionalEmbedding,
        "TransformerBlock": TransformerBlock
    },
    compile=False
)

print("Model loaded ✅")

Model loaded ✅


In [42]:
model.save("models/tuneformer_piano.keras")

In [43]:
import os, pickle, numpy as np, random

# Load vocab
with open("data/vocab.pkl", "rb") as f:
    pitchnames = pickle.load(f)

int_to_note = {i: n for i, n in enumerate(pitchnames)}
vocab_size = len(pitchnames)

# Load val for seed
X_val = np.load("data/X_val.npy").astype("int32")

def sample_with_temperature(probs, temperature=0.9):
    probs = np.asarray(probs).astype("float64")
    probs = np.log(probs + 1e-9) / temperature
    probs = np.exp(probs)
    probs = probs / np.sum(probs)
    return np.random.choice(len(probs), p=probs)

def generate_token_ids(model, seed, gen_len=800, temperature=0.9, seq_len=128):
    generated = list(seed.tolist())
    for _ in range(gen_len):
        x_in = np.array(generated[-seq_len:], dtype="int32").reshape(1, seq_len)
        probs = model.predict(x_in, verbose=0)[0]
        next_id = sample_with_temperature(probs, temperature=temperature)
        generated.append(int(next_id))
    return generated[seq_len:]  # only new tokens

seed = X_val[random.randint(0, len(X_val)-1)]
token_ids = generate_token_ids(model, seed, gen_len=1200, temperature=0.85)

print("Generated piano tokens:", len(token_ids))

Generated piano tokens: 1200


In [44]:
import pretty_midi

def tokens_to_piano_midi(token_ids, bpm=120, duration_sec=60, filename="outputs/piano.mid"):
    os.makedirs("outputs", exist_ok=True)

    pm = pretty_midi.PrettyMIDI(initial_tempo=bpm)
    piano = pretty_midi.Instrument(program=pretty_midi.instrument_name_to_program("Acoustic Grand Piano"))

    t = 0.0
    for tid in token_ids:
        tok = int_to_note[int(tid)]
        pitch_str, dur_str = tok.split("_")
        pitch = int(pitch_str)
        dur = float(dur_str)

        # stop when we reach desired duration
        if t + dur > duration_sec:
            break

        piano.notes.append(pretty_midi.Note(
            velocity=90,
            pitch=pitch,
            start=t,
            end=t + dur
        ))
        t += dur

    pm.instruments.append(piano)
    pm.write(filename)
    return filename, t

piano_path, actual_len = tokens_to_piano_midi(
    token_ids,
    bpm=120,          # <-- user input
    duration_sec=60,  # <-- user input
    filename="outputs/piano.mid"
)

print("Piano MIDI saved:", piano_path, "| length:", round(actual_len, 2), "sec")

Piano MIDI saved: outputs/piano.mid | length: 60.0 sec


In [45]:
def add_drums_to_midi(piano_midi_path, bpm=120, duration_sec=60, drop_time_sec=30, out_path="outputs/piano_drums.mid"):
    pm = pretty_midi.PrettyMIDI(piano_midi_path)
    pm.initial_tempo = bpm

    # Drums must be is_drum=True (GM drum channel)
    drums = pretty_midi.Instrument(program=0, is_drum=True)

    sec_per_beat = 60.0 / bpm
    beats_total = int(duration_sec / sec_per_beat)

    # GM drum pitches
    KICK = 36
    SNARE = 38
    HAT_CLOSED = 42

    t = 0.0
    for beat in range(beats_total):
        t = beat * sec_per_beat

        # "drop" section = more energy
        in_drop = t >= drop_time_sec

        # Kick pattern
        kick_beats = [0, 2] if not in_drop else [0, 1.5, 2, 3.5]  # extra kicks in drop
        for kb in kick_beats:
            kt = (beat + kb) * sec_per_beat
            if kt < duration_sec:
                drums.notes.append(pretty_midi.Note(velocity=110, pitch=KICK, start=kt, end=kt + 0.08))

        # Snare on 2 & 4
        for sb in [1, 3]:
            st = (beat + sb) * sec_per_beat
            if st < duration_sec:
                drums.notes.append(pretty_midi.Note(velocity=105, pitch=SNARE, start=st, end=st + 0.08))

        # Hi-hat 8th notes (more dense after drop)
        hat_step = 0.5 if not in_drop else 0.25
        hh = 0.0
        while hh < 4:
            ht = (beat + hh) * sec_per_beat
            if ht < duration_sec:
                drums.notes.append(pretty_midi.Note(velocity=75 if not in_drop else 85,
                                                    pitch=HAT_CLOSED, start=ht, end=ht + 0.05))
            hh += hat_step

    pm.instruments.append(drums)
    pm.write(out_path)
    return out_path

final_path = add_drums_to_midi(
    piano_midi_path="outputs/piano.mid",
    bpm=120,             # user input
    duration_sec=60,     # user input
    drop_time_sec=30,    # user input (drop at 30s)
    out_path="outputs/piano_drums.mid"
)

print("Final Piano+Drums MIDI saved ✅:", final_path)

Final Piano+Drums MIDI saved ✅: outputs/piano_drums.mid


In [46]:
import os
print(os.listdir("outputs"))

['piano.mid', 'piano_drums.mid']
